### MLA from Scratch

Initial setup and parameters
* Embedding dimension, d_model = 8
* KV Cache dimension, kv_latent_dim = 4
* Number of heads, n_heads = 2
* Head Dimension, dh = d_model/n_heads = 8/2 ==> 4
* Input token embedding, x = [0.1, -0.2, 0.3, -0.4, 0.5, -0.6, 0.7, 0.8]
* Prior context, 5 tokens ("The next day is bright")

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [10]:
class RopelessMLA(nn.Module):
    def __init__(self, d_model, n_heads, kv_latent_dim):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.dh = d_model//n_heads #dimenson per head

        # Projection layers
        self.W_q = nn.Linear(d_model, d_model, bias = False) # Query projection (input embedding dimension, output embedding dimension)
        self.W_dkv = nn.Linear(d_model, kv_latent_dim, bias = False) # Latent space (input embedding dimension, latent dimension)
        self.W_uk = nn.Linear(kv_latent_dim, d_model, bias = False) # Up keys (latent dimension, output dimension of the model)
        self.W_uv = nn.Linear(kv_latent_dim, d_model, bias = False) # Up values (latent dimension, output dimension of the model)
        self.W_o = nn.Linear(d_model, d_model, bias = False) # Output projection ()

        self.ln = nn.LayerNorm(kv_latent_dim) #layer normalization layer
        self.register_buffer('absorbed_k', None) #holds W_q @ W_uk

    def forward(self, x, kv_cache = None, past_length = 0):
        B, S, D = x.size() # 1, 1, 8

        # compute absorbed_k: W_q @ W_uk , shape(input embeddding, latent dim)
        if self.absorbed_k is None:
            absorbed = torch.matmul(self.W_q.weight, self.W_uk.weight) # (D, latent_dim)
            self.absorbed_k = absorbed.view(self.n_heads, self.dh, -1) # (n_heads, dh, latent_dim --> 2, 4, 4) group by heads

        # compress x into latent KV space
        new_c_kv = self.ln(self.W_dkv(x)) # (B, S, latent_dim) Finding the latent kv vector for the new input token
        if kv_cache is None:
            c_kv = new_c_kv
        else:
            c_kv = torch.cat([kv_cache, new_c_kv], dim=1) # (B, S total, latent_dim) appending the new kv vector to the kv cache
        
        S_full = c_kv.size(1)

        # Decompress V to full d_model
        v_full = self.W_uv(c_kv) # (B, S_full, D) multiplying c_kv and W_uv to get values matrix
        v = v_full.view(B, S_full, self.n_heads, self.dh).transpose(1, 2) # (B, n_heads, S_full, dh) it is grouped by heads or splitted by heads

        # splitting input by the number of heads
        q = x.view(B, S, self.n_heads, self.dh) # (B, S, n_heads, dh) #splitting happens over here

        # computing attention scores
        attn_scores = torch.zeros(B, self.n_heads, S, S_full, device = x.device)

        for h in range(self.n_heads):
            tmp = torch.matmul(q[:,:,h], self.absorbed_k[h]) #(B, S, latent_dim) muliplyinh the q(head0) with the first part of the self abosrbed matrix--> getting the tmp for each head
            attn_scores[:, h] = torch.bmm(tmp, c_kv.transpose(1,2)) #(B, S, S_full) multiplying the tmp with new cache transpose 

        # scaling and applying causal mask
        attn_scores = attn_scores / (self.dh**0.5) #dividing by the sqrt of head dimension
        mask = torch.tril(torch.ones((S, S_full), device = x.device), diagonal= past_length)
        attn_scores = attn_scores.masked_fill(mask.view(1, 1, S, S_full) ==0 , float('-inf'))

        # Softmax to get attention weights
        attn_weights = F.softmax(attn_scores, dim = -1) # (B, n_heads, S, S_full)

        # Applying attention weights to each V head separately
        out_heads = []
        for h in range(self.n_heads):
            context_h = torch.matmul(attn_weights[:, h], v[:, h]) # (B, S, dh) multiplying the attention weight sof each head with the value maxtrix of the head
            out_heads.append(context_h) #combining all the context vector
        
        # concatenate all heads outputs along the feature dimension
        out = torch.cat(out_heads, dim=-1) # (B, S, D)

        return self. W_o(out), c_kv # Final output projection + updated latent cache



Speed Testing

In [13]:
def demo():
    model = RopelessMLA(d_model = 512, n_heads =8, kv_latent_dim=256)
    x = torch.randn(1 ,5 ,512) # batch = 2, Sequence = 10, d_model = 512

    out, cache = model(x)
    print(f"Output: {out.shape}, Cache: {cache.shape}")

    # Memory comparison
    std_size = 2 * 2 * 10 * 512 * 4 / 1024 # KB (standard KV: B * 2 (K,V) * T *D * float32)
    latent_size = 2* 10* 256 * 4 / 1024 # KB (latent cache: B * T * latent_dim * float32)
    print(f"Memory: Standard = {std_size:.1f}KB, Latent = {latent_size:.1f}KB, Reduction = {std_size/latent_size:.1f}x ")

if __name__ == "__main__":
    demo()

Output: torch.Size([1, 5, 512]), Cache: torch.Size([1, 5, 256])
Memory: Standard = 80.0KB, Latent = 20.0KB, Reduction = 4.0x 


Cache testing - Single new inference

In [ ]:
def demo_cache_usage():
    torch.manual_seed(0)

    model = RopelessMLA(d_model = 8, n_heads =2, kv_latent_dim=4)

    # Initial input (sequence of 5 tokens)
    x1 = torch.randn(1, 5, 8) # (B=1, S=5, D=8)
    out1, cache1 = model(x1)

    print("Step 1: Intial input")
    print(f"Output shape:{out1.shape}")
    print(f"Cache shape: {cache1.shape}") #expect (1,5,4)

    # Append 1 token
    x2 = torch.rand(1, 1, 8) #(B=1, s =1 , D=8)
    out2, cache2 = model(x2, kv_cache = cache1, past_length =5)

    print("\nStep 2: New token")
    print(f"Output Shape: {out2.shape}")
    print(f"Cache Shape: {cache2.shape}") #expected (1,6,4)

if __name__ == "__main__":
    demo_cache_usage()


Step 1: Intial input
Output shape:torch.Size([1, 5, 8])
Cache shape: torch.Size([1, 5, 4])

Step 2: New token
Output Shape: torch.Size([1, 1, 8])
Cache Shape: torch.Size([1, 6, 4])


Cache testing multiple new inference 

In [16]:
def demo_kv_cache_growth(num_initial_tokens = 5, num_new_tokens = 3):
    torch.manual_seed(0)

    model = RopelessMLA(d_model = 8, n_heads =2, kv_latent_dim=4)

    # Start with intial token batch
    x = torch.randn(1, num_initial_tokens, 8)
    out, cache = model(x)
    print(f"Step 0: Intial input of {num_initial_tokens} tokens --> cache shape: {cache.shape}")

    # Incrementally append new tokens one at a time
    for step in range(1, num_new_tokens + 1):
        new_token = torch.randn(1, 1, 8) #(B=1, S=1, D=8)
        out, cache= model(new_token, kv_cache = cache,past_length = cache.shape[1])
        print(f"Step {step}: Added 1 token --> cache shape: {cache.shape}")

demo_kv_cache_growth(num_initial_tokens = 50, num_new_tokens = 4)
    

Step 0: Intial input of 50 tokens --> cache shape: torch.Size([1, 50, 4])
Step 1: Added 1 token --> cache shape: torch.Size([1, 51, 4])
Step 2: Added 1 token --> cache shape: torch.Size([1, 52, 4])
Step 3: Added 1 token --> cache shape: torch.Size([1, 53, 4])
Step 4: Added 1 token --> cache shape: torch.Size([1, 54, 4])
